# Escalado de datos y predicción del precio de un coche
**Objetivo:** predecir el **precio** de un coche a partir de su **potencia (CV)** y sus **años**.

Aprenderemos:
1. Por qué hay que escalar los datos.
2. Cómo escalar **sin hacer trampas** (evitar la *fuga de datos* o *data leakage*).
3. Cómo escalar un coche **nuevo** antes de predecir.
4. La forma **recomendada**: `Pipeline`, que hace todo automáticamente y evita errores.

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

## 2. Crear un conjunto de datos de ejemplo
Generamos 200 coches inventados. La regla "real" que se esconde en los datos es:

**precio ≈ 8.000 + 120 · potencia − 1.500 · años + ruido**

In [ ]:
np.random.seed(42)
n = 200

potencia = np.random.randint(60, 300, n)      # caballos (CV)
anios    = np.random.randint(0, 20, n)        # antigüedad en años
ruido    = np.random.normal(0, 2000, n)

precio = 8000 + 120 * potencia - 1500 * anios + ruido
precio = np.clip(precio, 1000, None)          # ningún coche por debajo de 1.000 €

df = pd.DataFrame({"potencia": potencia, "anios": anios, "precio": precio.round(0)})
df.head()

In [ ]:
df.describe().round(1)

### ¿Por qué escalar?
Mira las escalas: la **potencia** va de 60 a 300, los **años** de 0 a 19.
Muchos modelos (KNN, SVM, redes neuronales, regresión con regularización…) calculan **distancias** o usan
**gradientes**, y la variable con números más grandes "manda" más aunque no sea más importante.

`StandardScaler` transforma cada columna para que tenga **media 0** y **desviación típica 1**:

$$z = \frac{x - \text{media}}{\text{desviación típica}}$$

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(df["potencia"], df["precio"], alpha=0.6)
ax[0].set_xlabel("Potencia (CV)"); ax[0].set_ylabel("Precio (€)"); ax[0].set_title("Precio vs potencia")
ax[1].scatter(df["anios"], df["precio"], alpha=0.6, color="tab:orange")
ax[1].set_xlabel("Años"); ax[1].set_title("Precio vs años")
plt.tight_layout(); plt.show()

## 3. Separar variables de entrada (X) y salida (y)

In [ ]:
X = df[["potencia", "anios"]]   # las DOS variables que vamos a escalar
y = df["precio"]                # lo que queremos predecir (no se escala)

## 4. Dividir en entrenamiento y prueba **ANTES** de escalar
⚠️ **Regla de oro:** primero se divide, después se escala.

Si escalamos con *todos* los datos, la media y la desviación típica incluyen información del conjunto de prueba.
El modelo "ha visto" algo de los datos de examen → la nota sale **más optimista** de lo real. Eso es la **fuga de datos**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

## 5. Escalado manual (paso a paso)
- `fit_transform` **solo** con `X_train` → aprende la media y la desviación típica del entrenamiento.
- `transform` con `X_test` → usa **las mismas** medias y desviaciones (¡no se vuelve a hacer `fit`!).

In [ ]:
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)   # aprende y transforma
X_test_esc  = scaler.transform(X_test)        # solo transforma

print("Medias aprendidas (potencia, años):     ", scaler.mean_.round(2))
print("Desv. típicas aprendidas (potencia, años):", scaler.scale_.round(2))

In [ ]:
# Comprobación: el entrenamiento queda con media ~0 y desviación ~1
pd.DataFrame(X_train_esc, columns=["potencia_esc", "anios_esc"]).describe().round(2)

## 6. Entrenar el modelo y evaluarlo

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)

pred_test = modelo.predict(X_test_esc)
print(f"Error medio absoluto (MAE): {mean_absolute_error(y_test, pred_test):,.0f} €")
print(f"R²: {r2_score(y_test, pred_test):.3f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred_test, alpha=0.7)
lim = [y_test.min(), y_test.max()]
plt.plot(lim, lim, "r--", label="Predicción perfecta")
plt.xlabel("Precio real (€)"); plt.ylabel("Precio predicho (€)"); plt.legend()
plt.title("Real vs predicho"); plt.show()

## 7. Predecir el precio de un coche NUEVO (escalando previamente)
El coche nuevo **debe escalarse con el mismo `scaler`** del entrenamiento.
Si no lo escalas, o lo escalas con un `StandardScaler` nuevo, la predicción será un disparate.

In [ ]:
coche_nuevo = pd.DataFrame({"potencia": [150], "anios": [5]})

coche_nuevo_esc = scaler.transform(coche_nuevo)     # ✅ mismo scaler, solo transform
print("Coche nuevo escalado:", coche_nuevo_esc.round(3))

precio_pred = modelo.predict(coche_nuevo_esc)[0]
print(f"Precio estimado: {precio_pred:,.0f} €")

In [ ]:
# ❌ ERROR TÍPICO: predecir sin escalar
precio_mal = modelo.predict(coche_nuevo.values)[0]
print(f"Sin escalar (MAL): {precio_mal:,.0f} €   <-- no tiene sentido")

## 8. ✅ La mejor forma: `Pipeline`
Un `Pipeline` une **escalador + modelo** en un solo objeto:
- Al hacer `fit`, el escalador aprende **solo** con los datos de entrenamiento.
- Al hacer `predict`, escala automáticamente los datos nuevos con los parámetros correctos.
- Con validación cruzada, el escalador se recalcula **dentro de cada pliegue** → no hay fuga de datos.
- Es imposible "olvidarse" de escalar un coche nuevo.

In [ ]:
pipe = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", LinearRegression())
])

pipe.fit(X_train, y_train)            # ¡le pasamos los datos SIN escalar!

pred_pipe = pipe.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred_pipe):,.0f} €   R²: {r2_score(y_test, pred_pipe):.3f}")

# Predicción del coche nuevo: también SIN escalar, el pipeline lo hace solo
print(f"Precio estimado coche nuevo: {pipe.predict(coche_nuevo)[0]:,.0f} €")

In [ ]:
# Varios coches a la vez
coches = pd.DataFrame({
    "potencia": [90, 150, 250, 300],
    "anios":    [12,   5,   2,   0]
})
coches["precio_estimado"] = pipe.predict(coches[["potencia", "anios"]]).round(0)
coches

### Validación cruzada sin fuga de datos
`cross_val_score` con el pipeline divide los datos en 5 partes; en cada vuelta el escalador
se ajusta solo con las 4 partes de entrenamiento.

In [ ]:
scores = cross_val_score(pipe, X, y, cv=5, scoring="r2")
print("R² en cada pliegue:", scores.round(3))
print(f"R² medio: {scores.mean():.3f}")

## 9. ¿Cuándo importa DE VERDAD el escalado?
La regresión lineal simple da casi el mismo resultado con o sin escalar (solo cambian los coeficientes).
Pero **KNN** usa distancias, y ahí el escalado es decisivo. Comparemos:

In [ ]:
knn_sin = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
knn_con = Pipeline([("escalador", StandardScaler()),
                    ("modelo", KNeighborsRegressor(n_neighbors=5))]).fit(X_train, y_train)

for nombre, m in [("KNN SIN escalar", knn_sin), ("KNN CON escalar", knn_con)]:
    p = m.predict(X_test)
    print(f"{nombre:16s} -> MAE: {mean_absolute_error(y_test, p):7,.0f} €   R²: {r2_score(y_test, p):.3f}")

## 10. Interpretar el modelo (coeficientes)
Con datos escalados, cada coeficiente indica cuánto cambia el precio al aumentar
**una desviación típica** esa variable → se pueden comparar entre sí.

In [ ]:
coefs = pipe.named_steps["modelo"].coef_
for nombre, c in zip(X.columns, coefs):
    print(f"{nombre:9s}: {c:+,.0f} € por cada desviación típica")

# Volver a unidades originales (€ por CV y € por año)
coef_orig = coefs / pipe.named_steps["escalador"].scale_
print("\nEn unidades originales:")
print(f"  potencia: {coef_orig[0]:+,.1f} € por CV   (real: +120)")
print(f"  años:     {coef_orig[1]:+,.1f} € por año  (real: -1.500)")

## 11. Guardar y cargar el modelo (con su escalado incluido)
Al guardar el pipeline, el escalador va dentro: no hay que guardar dos archivos.

In [ ]:
import joblib
joblib.dump(pipe, "modelo_precio_coches.joblib")

pipe_cargado = joblib.load("modelo_precio_coches.joblib")
print(f"Predicción con modelo cargado: {pipe_cargado.predict(coche_nuevo)[0]:,.0f} €")

## 📌 Resumen
| ✅ Hacer | ❌ No hacer |
|---|---|
| Dividir train/test **antes** de escalar | Escalar todo el dataset y luego dividir |
| `fit_transform` en train, `transform` en test | Hacer `fit` también con test |
| Escalar el coche nuevo con **el mismo** scaler | Predecir sin escalar o con un scaler nuevo |
| Usar `Pipeline` (escalador + modelo) | Gestionar escalador y modelo por separado |
| No escalar la variable objetivo `y` (aquí no hace falta) | — |

## 🧪 Ejercicios
1. Cambia `StandardScaler` por `MinMaxScaler` y compara los resultados.
2. Añade una tercera variable (por ejemplo `km`) al dataset y al pipeline.
3. Prueba KNN con `n_neighbors` = 3, 10 y 20. ¿Cuál funciona mejor?
4. Haz el escalado "mal" (antes de dividir) y compara el R² con validación cruzada.